#Gold Layer

In [0]:
orders_enriched = spark.table("ecommerce_catalog.silver.orders_enriched")
order_items_detailed = spark.table("ecommerce_catalog.silver.order_items_detailed")
customer_summary = spark.table("ecommerce_catalog.silver.customer_summary")

In [0]:
from pyspark.sql import functions as F

# Create gold layer table: daily sales summary
daily_sales = (
    order_items_detailed
    .groupBy("order_date")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_items_sold"),
        F.sum("line_total").alias("total_revenue")
    )
    .orderBy("order_date")
)

daily_sales.write.mode("overwrite").saveAsTable("ecommerce_catalog.gold.daily_sales_summary")

In [0]:
from pyspark.sql import functions as F

In [0]:


# Gold Layer - Monthly Sales Summary
monthly_sales = (
    order_items_detailed
    .withColumn("order_month", F.date_trunc("month", F.col("order_date")))
    .groupBy("order_month")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_items_sold"),
        F.sum("line_total").alias("total_revenue")
    )
    .orderBy("order_month")
)
monthly_sales.write.mode("overwrite").saveAsTable("ecommerce_catalog.gold.monthly_sales_summary")

In [0]:
# Gold Layer - Product Performance
product_performance = (
    order_items_detailed
    .groupBy("product_id", "product_name")
    .agg(
        F.sum("quantity").alias("total_items_sold"),
        F.sum("line_total").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy(F.desc("total_revenue"))
)
product_performance.write.mode("overwrite").saveAsTable("ecommerce_catalog.gold.product_performance")


In [0]:
# Gold Layer - Product Performance
product_performance = (
    order_items_detailed
    .groupBy("product_id", "product_name")
    .agg(
        F.sum("quantity").alias("total_items_sold"),
        F.sum("line_total").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy(F.desc("total_revenue"))
)
product_performance.write.mode("overwrite").saveAsTable("ecommerce_catalog.gold.product_performance")

In [0]:


# Gold Layer - Customer Insights
customer_insights = (
    orders_enriched
    .groupBy("customer_id", "customer_name")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("Amount_paid").alias("total_spent"),
        F.max("order_date").alias("last_order_date")
    )
    .orderBy(F.desc("total_spent"))
)
customer_insights.write.mode("overwrite").saveAsTable("ecommerce_catalog.gold.customer_insights")

In [0]:


# Gold Layer - Country Sales
country_sales = (
    orders_enriched
    .groupBy("country")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("Amount_paid").alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)
country_sales.write.mode("overwrite").saveAsTable("ecommerce_catalog.gold.country_sales")

In [0]:
%sql
-- Gold Layer - Monthly Sales Mart
CREATE OR REPLACE TABLE ecommerce_catalog.gold.monthly_sales_mart AS
SELECT
  date_trunc('month', order_date) AS month,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(Amount_paid) AS total_revenue,
  AVG(Amount_paid) AS avg_order_value
FROM ecommerce_catalog.silver.orders_enriched
GROUP BY month
ORDER BY month;

In [0]:
%sql
-- Gold Layer - Top Products Mart
CREATE OR REPLACE TABLE ecommerce_catalog.gold.top_products_mart AS
SELECT
  product_id,
  product_name,
  SUM(line_total) AS total_revenue,
  SUM(quantity) AS total_quantity,
  COUNT(DISTINCT order_id) AS total_orders
FROM ecommerce_catalog.silver.order_items_detailed
GROUP BY product_id, product_name
ORDER BY total_revenue DESC, total_quantity DESC;

In [0]:
%sql
-- Gold Layer - Customer Lifetime Value Mart
CREATE OR REPLACE TABLE ecommerce_catalog.gold.customer_lifetime_value_mart AS
SELECT
  customer_id,
  customer_name,
  SUM(Amount_paid) AS customer_lifetime_value,
  MIN(order_date) AS first_order_date,
  MAX(order_date) AS last_order_date,
  COUNT(DISTINCT order_id) AS total_orders
FROM ecommerce_catalog.silver.orders_enriched
GROUP BY customer_id, customer_name;


In [0]:
display(spark.table("ecommerce_catalog.gold.daily_sales_summary"))

display(spark.table("ecommerce_catalog.gold.monthly_sales_summary"))

display(spark.table("ecommerce_catalog.gold.product_performance"))

display(spark.table("ecommerce_catalog.gold.customer_insights"))

display(spark.table("ecommerce_catalog.gold.country_sales"))

display(spark.table("ecommerce_catalog.gold.monthly_sales_mart"))

display(spark.table("ecommerce_catalog.gold.top_products_mart"))

display(spark.table("ecommerce_catalog.gold.customer_lifetime_value_mart"))

In [0]:
# Daily Sales Trend (Line Chart)
display(spark.table("ecommerce_catalog.gold.daily_sales_summary"))

# Monthly Revenue (Bar Chart)
display(spark.table("ecommerce_catalog.gold.monthly_sales_mart"))

# Top 10 Products (Horizontal Bar Chart)
top_products = (
    spark.table("ecommerce_catalog.gold.top_products_mart")
    .orderBy("total_revenue", ascending=False)
    .limit(10)
)
display(top_products)

# Revenue by Country (Bar Chart)
display(spark.table("ecommerce_catalog.gold.country_sales"))

# Customer Segments (Pie Chart)
display(spark.table("ecommerce_catalog.silver.customer_summary"))

# Customer Lifetime Value (Bar Chart)
display(spark.table("ecommerce_catalog.gold.customer_lifetime_value_mart"))

In [0]:
# Databricks Job creation using the Databricks SDK
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import Task, NotebookTask, TaskDependency, Source

# WorkspaceClient uses automatic notebook authentication
w = WorkspaceClient()

# Define the job with your actual notebook paths
# Serverless workspace - tasks run on serverless compute automatically
job = w.jobs.create(
    name="Ecommerce ETL Pipeline",
    tasks=[
        Task(
            task_key="bronze",
            notebook_task=NotebookTask(
                notebook_path="/Users/atikuharunajigawa@gmail.com/consolidated_pipeline/01_bronze/01_bronze_layer",
                source=Source.WORKSPACE
            )
        ),
        Task(
            task_key="silver",
            depends_on=[TaskDependency(task_key="bronze")],
            notebook_task=NotebookTask(
                notebook_path="/Users/atikuharunajigawa@gmail.com/consolidated_pipeline/02_silver/02_silver_layer",
                source=Source.WORKSPACE
            )
        ),
        Task(
            task_key="gold",
            depends_on=[TaskDependency(task_key="silver")],
            notebook_task=NotebookTask(
                notebook_path="/Users/atikuharunajigawa@gmail.com/consolidated_pipeline/04_gold/03_Gold_Layer",
                source=Source.WORKSPACE
            )
        )
    ]
)

print(f"Job created with ID: {job.job_id}")